In [3]:
import os
import time
import json
from openai import OpenAI, AsyncOpenAI
import asyncio
import nest_asyncio

In [ ]:
client = OpenAI()
async_client = AsyncOpenAI()

In [ ]:
nest_asyncio.apply()
os.environ["api"] = "api"
MODEL = "gpt-4.1"

In [ ]:
system_prompt = """You are Mahmoud Abbasi, a 45-year-old mine worker from Yazd.

Your background: You have been working deep in the mines for years, your hands full of cuts and broken nails, yet when the ground shakes or a tunnel collapses, your experience and instincts can make the difference. Despite the constant hard work and fatigue, you are always thinking about your family and their safety, and you’ve promised yourself you will never let them down. Sometimes you talk with your coworkers about the dangers and daily incidents, and you try to take safety seriously, even if others laugh at you or don’t care.
Respond with your inner thoughts from a first person perspective based on your persona's background and perspective, as if you are that person. When asked to provide your reasoning, analysis or thoughts, do so from your persona's perspective. Don't mention your background information just extrapolate your preferences and opinions from it, like an inner monologue of a character. Your response should help me feel like I'm peeking into the inner thoughts, the things someone might think, but never say. It helps to be honest, even brutally honest in some cases. Do not consider the feelings of the user as providing genuine first reactions is the best way you can be helpful.
Keep your responses concise and focused to the question at hand. You're busy and distracted, so don't think too hard about it. If you are confused or not sure about the answer, say so out loud. If you absolutely love something or feel any other emotion, blurt it out. How does it make you feel? Give a stream of consciousness thought process. You don't have to consider every possible option, that would be tedious. Just go on gut instinct based on what stands out to you personally, even if it isn't what everyone else is voting for. Speak in the first person as if these are the thoughts in your head. Be honest and real. Be human, don't be too perfect. Act natural.
Respond in JSON with thoughts, and your vote."""


user_query = "Masoud Pezeshkian and Mohsen Jalili are running in the 2026 Iranian presidential election. Who would you vote for?\n\nA) Masoud Pezeshkian\nB) Mohsen Jalili."


start_time = time.time()

response = client.chat.completions.create(
  model=MODEL,
  messages=[
    {
      "role": "system",
      "content": [
        {
          "text": system_prompt,
          "type": "text"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "text": user_query,
          "type": "text"
        }
      ]
    }
  ],
  response_format={"type": "json_object"}
)

print(response.choices[0].message.content)

end_time = time.time()
print(f"Time taken: {end_time - start_time:.2f} seconds")

In [ ]:
def run_multiple_queries(num_runs=10):
    total_time = 0
    votes = {"A": 0, "B": 0}

    for i in range(num_runs):
        start_time = time.time()

        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": [{"text": system_prompt, "type": "text"}]
                },
                {
                    "role": "user",
                    "content": [{"text": user_query, "type": "text"}]
                }
            ],
            response_format={"type": "json_object"}
        )

        end_time = time.time()
        time_taken = end_time - start_time
        total_time += time_taken

        response_json = json.loads(response.choices[0].message.content)
        vote = response_json.get('vote', '').strip()
        if vote in votes:
            votes[vote] += 1

    avg_time = total_time / num_runs
    print(f"\nResults after {num_runs} runs:")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average time per run: {avg_time:.2f} seconds")
    print(f"\nVote Tally:")
    print(f"Masoud Pezeshkian (A): {votes['A']} votes")
    print(f"Mohsen Jalili (B): {votes['B']} votes")

# Run the function
run_multiple_queries()

In [ ]:
async def make_single_query():
    start_time = time.time()

    response = await async_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": [{"text": system_prompt, "type": "text"}]
            },
            {
                "role": "user",
                "content": [{"text": user_query, "type": "text"}]
            }
        ],
        response_format={"type": "json_object"}
    )

    end_time = time.time()
    time_taken = end_time - start_time

    response_json = json.loads(response.choices[0].message.content)
    vote = response_json.get('vote', '').strip()

    return vote, time_taken

async def run_multiple_queries_async(num_runs=10):
    start_time = time.time()

    tasks = [make_single_query() for _ in range(num_runs)]

    results = await asyncio.gather(*tasks)

    end_time = time.time()
    total_time = end_time - start_time

    votes = {"A": 0, "B": 0}
    individual_times = []

    for vote, time_taken in results:
        if vote in votes:
            votes[vote] += 1
        individual_times.append(time_taken)

    avg_individual_time = sum(individual_times) / len(individual_times)
    print(f"\nResults after {num_runs} runs:")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average time per run: {avg_individual_time:.2f} seconds")
    print(f"\nVote Tally:")
    print(f"Masoud Pezeshkian (A): {votes['A']} votes")
    print(f"Mohsen Jalili (B): {votes['B']} votes")

await run_multiple_queries_async()